# Como a ferramenta funciona — passo a passo com 10 anúncios

Este notebook explica, de forma simples, **o que estamos buscando**, **onde** e **como**, e mostra cada etapa
funcionando com dados reais.

**O objetivo do projeto:** estudar os anúncios de pessoas escravizadas publicados nos jornais do século XIX
(fuga, compra, venda, aluguel e ama de leite) e ver como eles mudam ao longo das leis que marcam o fim da
escravidão:

| Ano | Lei | O que mudou |
|---|---|---|
| 1831 | Lei Feijó | proíbe (no papel) o tráfico de africanos |
| 1850 | Lei Eusébio de Queirós | acaba de fato com o tráfico atlântico |
| 1871 | Lei do Ventre Livre | filhos de escravizadas nascem livres |
| 1885 | Lei dos Sexagenários | liberta os maiores de 60 anos |
| 1888 | Lei Áurea | abolição |

Para isso precisamos transformar **páginas de jornal** em uma **tabela de anúncios**, com uma linha por
anúncio e colunas como tipo, sexo, idade, nação e período legal. É isso que a ferramenta `hdb` faz.

**O caminho, em 6 passos:**

```
1. Onde procurar   →  2. Buscar páginas  →  3. Capturar e ler (OCR)
4. Separar anúncios →  5. Extrair campos  →  6. Analisar por período legal
```

> Como rodar: abra este notebook com o ambiente do projeto (`.venv`). As células que leem o banco precisam do
> arquivo `.env` (o mesmo usado pela ferramenta). Nenhuma célula mostra a chave secreta.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from hdb import config, busca, campos
from hdb.banco import BancoMemoria, BancoSupabase

pd.set_option("display.max_colwidth", 120)
banco = BancoSupabase()          # o banco compartilhado do grupo (lê o .env)
db = banco.cliente               # atalho para consultas
print("Fonte dos dados:", config.HDB_URL)

Fonte dos dados: https://memoria.bn.gov.br/hdb/


## A fonte: Hemeroteca Digital Brasileira (HDB)

A HDB, da Biblioteca Nacional, tem milhares de jornais digitalizados. Cada jornal tem um código, o **`bib`**.
Por exemplo, `364568_04` é o *Jornal do Commercio* de 1850 a 1859.

A busca da HDB tem algumas regras importantes:

- ela devolve **páginas**, não anúncios: "671 ocorrências" quer dizer 671 páginas onde o termo aparece;
- aspas buscam a **frase exata** (`"escravo fugido"`), e `*` no fim aceita variações (`escrav*` acha escravo,
  escrava, escravos…);
- maiúsculas e acentos não importam.

Por isso escolhemos termos que apontam direto para os anúncios que queremos, como `"escrav* fugid*"`,
`"vende-se uma preta"` e `"preta com leite"` (a lista completa está em `termos_busca.md`).

## Passo 1 — Onde procurar: o panorama

Antes de buscar, perguntamos: **em quais jornais o termo mais aparece?** Os comandos `hdb acervos` e
`hdb panorama` descobriram os 186 periódicos do Rio de Janeiro de 1850–1859 e contaram em cada um as páginas com
`"escrav* fugid*"`. O resultado está no banco, na view `panorama`:

In [2]:
termo = '"escrav* fugid*"'
linhas = (db.table("panorama").select("nome,bib,ano_inicio,ano_fim,paginas,total_paginas,percentual,cobre_mais_que_a_decada")
          .eq("local", "RJ").eq("periodo", "1850 - 1859").eq("termo", termo)
          .gt("paginas", 0).order("paginas", desc=True).limit(10).execute().data)
panorama = pd.DataFrame(linhas)
panorama["percentual"] = (panorama["percentual"].astype(float) * 100).round(2)
panorama

,nome,bib,ano_inicio,ano_fim,paginas,total_paginas,percentual,cobre_mais_que_a_decada
0,Diário do Rio de Janeiro (RJ) - 1821 a 1858,094170_01,1821,1858,3464,46933,7.38,True
1,"Correio Mercantil, e Instructivo, Politico, Universal (RJ) - 1848 a 1868",217280,1848,1868,1176,29822,3.94,True
2,Jornal do Commercio (RJ) - 1850 a 1859,364568_04,1850,1859,671,15569,4.31,False
3,Revista do Instituto Histórico e Geográfico Brasileiro (RJ) - 1839 a 2021,893676,1839,2021,131,201064,0.07,True
4,"O Correio da Tarde : Jornal Commercial, Politico, Litterario e Noticioso (RJ) - 1855 a 1862",090000,1855,1862,110,7831,1.40,True
5,A Patria : Folha da Provincia do Rio de Janeiro (RJ) - 1856 a 1889,830330,1856,1889,88,13975,0.63,True
6,Annaes do Parlamento Brasileiro (RJ) - 1826 a 1888,132489,1826,1888,67,94212,0.07,True
7,Brasil Commercial : Propriedade de uma Associação (RJ) - 1858,810282,1858,1858,43,506,8.50,False
8,O Parahyba (RJ) - 1857 a 1859,809047,1857,1859,24,824,2.91,False
9,"A Actualidade : jornal politico, litterario e noticioso (RJ) - 1859 a 1864",235296,1859,1864,22,2656,0.83,True


**Como ler:** `paginas` é quantas páginas têm o termo; `total_paginas` é o tamanho do acervo; `percentual` é a
proporção. Quando `cobre_mais_que_a_decada` é verdadeiro, o acervo inclui anos fora de 1850–1859 (o Diário do Rio
vai de 1821 a 1858), então a contagem não é só da década.

**Insight:** de 186 periódicos, poucos concentram quase todos os anúncios de fuga. É neles que vale gastar o
tempo de captura.

## Passo 2 — Buscar as páginas (10 ocorrências, ao vivo)

Agora buscamos o termo no *Jornal do Commercio* (`364568_04`) e pegamos as **10 primeiras ocorrências**.
Isso usa só requisições simples à HDB (sem navegador e sem CAPTCHA). Para cada ocorrência guardamos onde ela
está: ano, edição, página e link.

> Aqui usamos um banco **em memória** (`BancoMemoria`), para o exemplo não mexer no banco do grupo.
> Na ferramenta real, o comando é `hdb buscar --bib 364568_04 --termo '"escrav* fugid*"' --limite 10`.

In [3]:
bib = "364568_04"
sessao = busca.nova_sessao()
memoria = BancoMemoria()
salvas, total = busca.coletar_ocorrencias(sessao, bib, termo, memoria, limite=10)
print(f"A HDB informa {total} páginas com {termo}; listamos {salvas}.")
linhas = []
for o in memoria.ocorrencias.values():
    p = memoria.paginas[o["pagina_id"]]
    linhas.append({"ocorrencia": o["posicao"], "ano": p.get("ano"), "edicao": p.get("edicao"),
                   "pagina": p.get("pagina"), "link": p.get("link")})
pd.DataFrame(linhas)

A HDB informa 671 páginas com "escrav* fugid*"; listamos 10.


,ocorrencia,ano,edicao,pagina,link
0,1,1850,00016,3,https://memoria.bn.gov.br/docreader/364568_04/63
1,2,1850,00023,6,https://memoria.bn.gov.br/docreader/364568_04/94
2,3,1850,00037,4,https://memoria.bn.gov.br/docreader/364568_04/162
3,4,1850,00059,4,https://memoria.bn.gov.br/docreader/364568_04/260
4,5,1850,00061,4,https://memoria.bn.gov.br/docreader/364568_04/270
5,6,1850,00109,1,https://memoria.bn.gov.br/docreader/364568_04/465
6,7,1850,00177,3,https://memoria.bn.gov.br/docreader/364568_04/775
7,8,1850,00181,4,https://memoria.bn.gov.br/docreader/364568_04/796
8,9,1850,00190,3,https://memoria.bn.gov.br/docreader/364568_04/843
9,10,1850,00231,3,https://memoria.bn.gov.br/docreader/364568_04/1055


**O retorno** é uma lista de páginas, ainda **sem texto**. A HDB não entrega o texto das páginas: ele
precisa ser lido da imagem. Esse é o próximo passo.

## Passo 3 — Capturar a imagem e ler o texto (OCR)

**OCR** (reconhecimento óptico de caracteres) é o programa que "lê" a imagem e devolve o texto. Usamos o
**Tesseract**, gratuito e local.

Para ver a imagem de uma página, a HDB pede um **CAPTCHA**. A ferramenta **nunca** tenta resolvê-lo:
uma pessoa digita o código, e daí em diante a ferramenta trabalha sozinha.

```bash
PYTHONPATH=src .venv/bin/python -m hdb capturar --bib 364568_04 --termo '"escrav* fugid*"' --max-paginas 10
```

O que acontece em cada página:

1. reserva a página no banco (duas pessoas nunca pegam a mesma);
2. abre o leitor, pede a imagem em alta resolução e baixa;
3. roda o OCR e grava o texto e a confiança no banco;
4. apaga a imagem (só o texto fica) e marca a página como `ok`, ou registra o erro.

Uma página já lida **nunca é capturada de novo**. Por isso, no exemplo abaixo usamos páginas que já estão no
banco. Veja como fica o texto bruto de uma delas:

In [4]:
pagina = (db.table("paginas").select("id,bib,ano,edicao,pagina,link").eq("bib", bib).eq("status", "ok")
          .order("id").limit(1).execute().data[0])
texto = db.table("textos_atuais").select("texto,confianca,motor").eq("pagina_id", pagina["id"]).execute().data[0]
print(pagina)
print(f"Motor: {texto['motor']} | confiança média: {texto['confianca']:.1f} | {len(texto['texto'])} caracteres\n")
print(texto["texto"][:1200])

{'id': 24, 'bib': '364568_04', 'ano': 1850, 'edicao': '00010', 'pagina': '4', 'link': 'https://memoria.bn.gov.br/docreader/364568_04/40'}
Motor: tesseract-5.3.4 por psm3-pagina-cinza | confiança média: 69.4 | 29129 caracteres

ONT CERA TSPALLIVEL

oa esmurana , QUENBNNNDO, cributama Moenonão Eru diiqunso. nas cuimas, o osnanimos

PENIS SSTEs

a forma de war deste

de secos e molhados, ou mesmo des-

Pedregulho m. 152 precisa-se de um bos ví-
1 RADICAL DE QEBRADURAS E ROTURAS EO d IEP

proximamente do Porto, que dé fiador á
padecerem de roturas (por mais antigas que elias sejão), e que quizerem ter uma | «ua conducia; na rua nova do Conde n. 6 ne digá onde é. |
om dita,, queição diriginos à rua dn (oianda alpes penha ALUGA-SE um perfeito cotinbeiro de forno e fogão; À GRANDE POSTA REAL
remedio, explica o . . Recomenda-se a To temoástros motua do Resarion. sé, rondar. MM DE S. PETERSBURGO,
escravos , € a todas as pessoas que soffrem esta terrivel . . onde elle só (trabalho arriscado e no

**Como ler:** o texto vem com erros de letra (o papel é velho e a grafia é do século XIX) e com tudo da
página junto: notícias, avisos, anúncios de casas e de escravizados. A **confiança** é a média que o Tesseract
dá a cada palavra (0 a 100). Ela não mede o acerto, mas ajuda a comparar páginas.

## Passo 4 — Separar os anúncios

Uma página tem várias colunas e dezenas de anúncios. O comando `hdb anuncios`:

- descobre as **colunas** pela posição das palavras;
- acha o **começo de cada anúncio** pelas palavras de abertura (VENDE-SE, ALUGA-SE, FUGIO, PRECISA-SE…), pelos
  avisos numerados ("34.") e pelos títulos de seção ("ESCRAVOS FUGIDOS");
- junta as linhas de cada anúncio e desfaz as palavras partidas no fim da linha ("escra-\nvo" → "escravo").

Vamos pegar **10 anúncios** já separados, 2 de cada tipo, de jornais de 1840 em diante:

In [5]:
dez = []
for tipo in ["fuga", "venda", "aluguel", "ama_de_leite", "compra"]:   # 2 de cada tipo, para variar
    dez += (db.table("anuncios_completos")
            .select("id,acervo,ano,pagina,titulo,abertura,texto,periodo_legal,link")
            .eq("versao", config.VERSAO_ANUNCIOS).eq("c_versao", config.VERSAO_CAMPOS)
            .eq("c_tipo", tipo).gte("ano", 1840).order("id").limit(2).execute().data)
anuncios = pd.DataFrame(dez)
anuncios[["id", "acervo", "ano", "periodo_legal", "texto"]]

,id,acervo,ano,periodo_legal,texto
0,5933,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"Desappareceu , no dia 7 do corrente, hum preto de nome Andre, com eara redonda, besxigoso , nariz ar hatado e sempre..."
1,5948,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"FUGIO hum molecote de 14 a 16 annos de idade, de nação Benguella, de nome Joaquim, levando vestido camiza de baeta a..."
2,5938,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),VENDE-SE a casa terrea da rua de S. Pedro n 300; trata-se na rua do Ouvidor n. 133 €. é — O BACIIAREL José Norberto ...
3,5951,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"VENDE-SE hum “preto cosinheiro, na rua do Cano n.78, que la acharão com quem tratar,"
4,5939,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"PRECISA-SE alugar hum preto para todo o serviço , na rua do Cano n. 23 A, podendo este ir dormir à casa de seu senho..."
5,5947,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),PRECISA-SE alugar huma rapariga perfeita costureira: na rua do Ouvidor n. 76.
6,5941,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"VENDE-SE, na rua do Hospicio n. 138, huma boa preta ama de leite, com cria de mez e meio , sabendo lavar e engommar,..."
7,5945,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),ALUGA-SE huma boa ama de leite : na rua dos Ourives n. 91. — A PESSOA que ganhou em rifa o riquissimo relogio de cim...
8,6034,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"PRECISA-SE comprar alguns pretos de roça velhos e de meia idade ; na rua da Misericordia n, 69. — J. R.8S. R. compro..."
9,6135,Jornal do Commercio (RJ) - 1840 a 1849,1840,1831–1850 (Feijó → Eusébio),"COMPRÃO-SE escravos com officios e sem elles, de ambos os sexos, ese pagão bem ; na rua dos Ourives, n. 17."


Cada linha agora é **um anúncio**, e não mais uma página inteira. Esse texto, anúncio por anúncio, é o que vai
para a **modelagem de tópicos**.

## Passo 5 — Extrair os campos

O comando `hdb campos` lê cada anúncio e procura, por regras de palavras:

| Campo | Exemplo de pista no texto |
|---|---|
| tipo | VENDE-SE → venda; FUGIO / alviçaras → fuga; "com leite" → ama de leite |
| sexo | preto/preta, moleque, negrinha, crioula… (só quando falam de uma pessoa) |
| idade | "de 13 annos", "de 16 a 18 annos" |
| nação | "de nação Congo", Mina, Benguela, Cabinda… |
| nome | "por nome Maria" |
| recompensa | gratifica, alviçaras, premio |

Para mostrar que isso não é mágica, rodamos a extração **aqui mesmo** sobre os 10 anúncios:

In [6]:
resultado = []
for a in dez:
    f = campos.extrair_anuncio(a)
    resultado.append({"id": a["id"], "tipo": f["tipo"], "sexo": f["sexo"],
                      "idade": f"{f['idade_min']}–{f['idade_max']}" if f["idade_min"] else None,
                      "nacoes": ", ".join(f["nacoes"]), "nomes": ", ".join(f["nomes"]),
                      "recompensa": f["recompensa"], "inicio do texto": a["texto"][:90]})
pd.DataFrame(resultado)

,id,tipo,sexo,idade,nacoes,nomes,recompensa,inicio do texto
0,5933,fuga,m,NaN,,Andre,True,"Desappareceu , no dia 7 do corrente, hum preto de nome Andre, com eara redonda, besxigoso"
1,5948,fuga,m,14–16,Benguela,Joaquim,False,"FUGIO hum molecote de 14 a 16 annos de idade, de nação Benguella, de nome Joaquim, levando"
2,5938,venda,m,NaN,Mina,,True,VENDE-SE a casa terrea da rua de S. Pedro n 300; trata-se na rua do Ouvidor n. 133 €. é —
3,5951,venda,m,NaN,,,False,"VENDE-SE hum “preto cosinheiro, na rua do Cano n.78, que la acharão com quem tratar,"
4,5939,aluguel,m,NaN,,,False,"PRECISA-SE alugar hum preto para todo o serviço , na rua do Cano n. 23 A, podendo este ir"
5,5947,aluguel,f,NaN,,,False,PRECISA-SE alugar huma rapariga perfeita costureira: na rua do Ouvidor n. 76.
6,5941,ama_de_leite,f,NaN,,,False,"VENDE-SE, na rua do Hospicio n. 138, huma boa preta ama de leite, com cria de mez e meio ,"
7,5945,ama_de_leite,f,NaN,,,False,ALUGA-SE huma boa ama de leite : na rua dos Ourives n. 91. — A PESSOA que ganhou em rifa o
8,6034,compra,m,NaN,,,False,PRECISA-SE comprar alguns pretos de roça velhos e de meia idade ; na rua da Misericordia n
9,6135,compra,m,NaN,,,False,"COMPRÃO-SE escravos com officios e sem elles, de ambos os sexos, ese pagão bem ; na rua do"


**Repare nos acertos e nos erros:**

- o anúncio de fuga do *molecote de 14 a 16 annos, de nação Benguella, de nome Joaquim* saiu completo: fuga,
  sexo masculino, idade 14–16, nação Benguela e nome Joaquim;
- já o "VENDE-SE a casa terrea…" saiu como **venda de pessoa**. O anúncio da casa ficou grudado no anúncio
  seguinte, que fala de um escravizado (Mina). O erro vem da separação dos anúncios (passo 4), não da extração.
  Por isso é preciso conferir uma amostra antes de usar os números.

Cada campo guarda a **evidência**, o trecho do texto que o justificou. Assim qualquer pessoa do grupo
pode conferir se a regra acertou:

In [7]:
exemplo = max(dez, key=lambda a: len(campos.extrair_anuncio(a)["evidencias"]))   # o mais rico em campos
f = campos.extrair_anuncio(exemplo)
print(exemplo["texto"][:500], "\n")
for campo, trecho in f["evidencias"].items():
    print(f"{campo:12} ← {trecho}")

Desappareceu , no dia 7 do corrente, hum preto de nome Andre, com eara redonda, besxigoso , nariz ar hatado e sempre elelo de tabaco, ea roupa bastante suja: - JM far, Quem o levar sabe ler, escrever e contar, 4) h rua do Ouvidor n, 65, recebera alviçaras, eme 

sexo         ← preto
cores        ← preto
nomes        ← Andre
recompensa   ← alviçara
endereco     ← rua do Ouvidor n, 65


## Passo 6 — Analisar pelo período legal

Com o ano de cada anúncio, o banco calcula o **período legal** (antes de 1831, 1831–1850, 1850–1871,
1871–1888). Agora dá para contar os tipos de anúncio em cada período, com todos os anúncios já
processados:

In [8]:
todos = []
inicio = 0
while True:   # a API devolve no máximo 1000 linhas por vez
    lote = (db.table("anuncios_completos").select("periodo_legal,c_tipo,c_sexo")
            .eq("versao", config.VERSAO_ANUNCIOS).eq("c_versao", config.VERSAO_CAMPOS)
            .neq("c_tipo", "outro").range(inicio, inicio + 999).execute().data)
    todos += lote
    if len(lote) < 1000:
        break
    inicio += 1000
df = pd.DataFrame(todos)
pd.crosstab(df["periodo_legal"], df["c_tipo"], margins=True, margins_name="total")

c_tipo,aluguel,ama_de_leite,compra,fuga,venda,total
periodo_legal,,,,,,
1831–1850 (Feijó → Eusébio),35,18,3,26,54,136
1850–1871 (Eusébio → Ventre Livre),68,18,2,32,38,158
1871–1888 (Ventre Livre → Áurea),3,0,0,18,4,25
antes de 1831,0,0,0,28,0,28
total,106,36,5,104,96,347


In [9]:
pd.crosstab(df["c_tipo"], df["c_sexo"].fillna("sem sexo"))

c_sexo,ambos,f,m,sem sexo
c_tipo,,,,
aluguel,8,49,44,5
ama_de_leite,3,33,0,0
compra,0,3,2,0
fuga,1,6,54,43
venda,15,37,44,0


**Como ler:** é aqui que a pergunta do projeto aparece. Por exemplo: os anúncios de **aluguel** aumentam
depois de 1850, quando o tráfico acaba? Os de **fuga** mudam depois de 1871?

**Cuidado:** por enquanto temos poucas páginas capturadas, escolhidas por termo e não sorteadas, e cada
período vem de jornais diferentes. Os números mostram **que a análise é possível**, mas ainda não permitem
conclusão histórica. Para isso falta capturar mais páginas, e o panorama do passo 1 diz quais.

## Resumo

| Passo | Comando | Entra | Sai |
|---|---|---|---|
| 1. Onde procurar | `hdb acervos`, `hdb panorama` | estado, década, termo | ranking de jornais |
| 2. Buscar | `hdb buscar` | jornal + termo | lista de páginas (sem texto) |
| 3. Capturar e ler | `hdb capturar` | páginas + pessoa no CAPTCHA | texto OCR de cada página |
| 4. Separar | `hdb anuncios` | texto + posição das palavras | um anúncio por linha |
| 5. Campos | `hdb campos` | anúncios | tipo, sexo, idade, nação, nome… |
| 6. Analisar | SQL / este notebook | tabela de anúncios | contagens por período legal |
| Método | `hdb relatorio` | o banco | relatório de como o corpus foi feito |

**Limites que convém lembrar:**

- o OCR erra letras, e algumas páginas saem ruins;
- às vezes dois anúncios vizinhos ficam juntos num só;
- os campos vêm de regras de palavras e ainda não têm uma taxa de acerto medida, então confira uma amostra;
- a captura depende de uma pessoa para o CAPTCHA.

Mais detalhes em `guia_ferramenta.md` (como usar) e `planejamento.md` (o que falta fazer).